# 🔵 Clustering com K-Means + Redução de Dimensionalidade (PCA)
**FIAP Postech — IA para Devs | Fase 1 · Machine Learning**

---

## O que você vai aprender
- Aprendizado **não-supervisionado**: não existe label/target
- Algoritmo **K-Means**: como funciona, como escolher K
- **Elbow Method** e **Silhouette Score** para validar K
- **PCA** (Principal Component Analysis): reduzir 30 features → 2 para visualizar
- Visualização de clusters em 2D com PCA
- Interpretação dos clusters encontrados

---
> **Supervised vs Unsupervised**  
> Supervisionado → você tem os rótulos, ensina o modelo (Ex: classificação, regressão)  
> **Não-supervisionado** → você NÃO tem rótulos, o modelo descobre padrões sozinho (Ex: clustering)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

import matplotlib.cm as cm

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ Imports OK')

---
## 1. Dataset
Usamos o Breast Cancer, mas **fingindo que não temos o label** — só veremos se o clustering acha os grupos ao final.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y_true = data.target  # guardado só para validação no final

print('Shape:', X.shape)
print('Classes reais (usaremos só no final para validar):', dict(zip([0,1], data.target_names)))
X.head()

In [ ]:
# Normalização — SEMPRE necessária antes do K-Means
# K-Means usa distância euclidiana: features em escalas diferentes distorcem os clusters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Antes do scaling — mean/std da primeira feature:')
print(f'  mean={X.iloc[:,0].mean():.2f}, std={X.iloc[:,0].std():.2f}')
print('\nApós StandardScaler:')
print(f'  mean={X_scaled[:,0].mean():.4f}, std={X_scaled[:,0].std():.4f}')

---
## 2. Como o K-Means Funciona

```
1. Escolha K centroides aleatórios
2. Atribua cada ponto ao centroide mais próximo (distância euclidiana)
3. Recalcule cada centroide como a média dos pontos do cluster
4. Repita 2-3 até convergir (centroides não mudam)
```

O objetivo é minimizar a **inércia** = soma das distâncias² de cada ponto ao seu centroide.

---
## 3. Elbow Method — Escolhendo o K ideal

In [ ]:
inercias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(K_range, inercias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(2, color='red', linestyle='--', alpha=0.7, label='K sugerido')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].set_title('Elbow Method\n(cotovelo = K ideal)', fontweight='bold')
axes[0].legend()
axes[0].set_xticks(list(K_range))

# Silhouette
axes[1].plot(K_range, silhouettes, 'gs-', linewidth=2, markersize=8)
best_k = list(K_range)[silhouettes.index(max(silhouettes))]
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Melhor K={best_k}')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score\n(quanto maior, melhor separação)', fontweight='bold')
axes[1].legend()
axes[1].set_xticks(list(K_range))

plt.suptitle('Escolha do K Ideal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n🏆 Melhor K pelo Silhouette: {best_k} (score={max(silhouettes):.3f})')

---
## 4. Aplicando K-Means com K=2

In [ ]:
km_final = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = km_final.fit_predict(X_scaled)

print(f'Silhouette Score (K=2): {silhouette_score(X_scaled, labels):.4f}')
print(f'Inércia: {km_final.inertia_:.2f}')
print(f'\nDistribuição dos clusters:')
unique, counts = np.unique(labels, return_counts=True)
for c, n in zip(unique, counts):
    print(f'  Cluster {c}: {n} amostras ({n/len(labels)*100:.1f}%)')

---
## 5. PCA — Redução de Dimensionalidade para Visualização

30 features → impossível visualizar. PCA comprime para 2 **componentes principais** que capturam a maior variância.

In [ ]:
# Variância explicada por cada componente
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

variancia_acumulada = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 11), pca_full.explained_variance_ratio_[:10] * 100,
            color='#3498db', edgecolor='white')
axes[0].set_title('Variância Explicada por Componente', fontweight='bold')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('% Variância Explicada')
axes[0].set_xticks(range(1, 11))

axes[1].plot(range(1, 11), variancia_acumulada[:10] * 100, 'bo-', linewidth=2)
axes[1].axhline(90, color='red', linestyle='--', alpha=0.7, label='90% de variância')
axes[1].set_title('Variância Acumulada', fontweight='bold')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('% Variância Acumulada')
axes[1].set_xticks(range(1, 11))
axes[1].legend()

plt.suptitle('PCA — Análise de Componentes Principais', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

n_para_90 = np.argmax(variancia_acumulada >= 0.90) + 1
print(f'\n📊 PC1 explica: {pca_full.explained_variance_ratio_[0]*100:.1f}%')
print(f'   PC2 explica: {pca_full.explained_variance_ratio_[1]*100:.1f}%')
print(f'   PC1+PC2 juntos: {sum(pca_full.explained_variance_ratio_[:2])*100:.1f}%')
print(f'   Para 90% da variância: {n_para_90} componentes (antes eram 30 features)')

In [ ]:
# Reduzindo para 2 componentes
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)

print(f'Shape original: {X_scaled.shape}')
print(f'Shape após PCA: {X_2d.shape}')
print(f'Variância preservada: {sum(pca_2d.explained_variance_ratio_)*100:.1f}%')

In [ ]:
# Visualização: clusters K-Means vs labels reais
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors_cluster = ['#e74c3c', '#3498db']
colors_true = ['#e74c3c', '#2ecc71']

# Clusters K-Means
for c in [0, 1]:
    mask = labels == c
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    alpha=0.5, s=25, color=colors_cluster[c], label=f'Cluster {c}')
# Centroides
centroides_2d = pca_2d.transform(km_final.cluster_centers_)
axes[0].scatter(centroides_2d[:, 0], centroides_2d[:, 1],
                s=200, marker='*', color='gold', edgecolors='black', zorder=5, label='Centroides')
axes[0].set_title('K-Means Clusters (K=2)', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend()

# Labels reais (para comparação)
for c, nome in zip([0, 1], data.target_names):
    mask = y_true == c
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    alpha=0.5, s=25, color=colors_true[c], label=nome)
axes[1].set_title('Labels Reais (maligno/benigno)', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].legend()

plt.suptitle('PCA 2D — K-Means vs Labels Reais', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Validando a Qualidade do Clustering

In [ ]:
# Adjusted Rand Index — compara clusters encontrados com labels reais
# (só possível porque temos os labels; na prática real, não teríamos)
ari = adjusted_rand_score(y_true, labels)
print(f'Adjusted Rand Index (ARI): {ari:.4f}')
print('  ARI = 1.0 → clusters idênticos aos labels reais')
print('  ARI = 0.0 → clustering aleatório')
print(f'\n  Nosso resultado: K-Means recuperou {ari*100:.1f}% da estrutura real sem ver os labels!')

# Comparação cluster vs label real
df_comp = pd.DataFrame({'Cluster KMeans': labels, 'Label Real': y_true})
print('\n📊 Tabela cruzada Cluster vs Label Real:')
print(pd.crosstab(df_comp['Cluster KMeans'], df_comp['Label Real'],
                  rownames=['Cluster'], colnames=['Real (0=maligno, 1=benigno)']))

In [ ]:
# Silhouette plot detalhado
sil_vals = silhouette_samples(X_scaled, labels)

fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10
colors_sil = ['#e74c3c', '#3498db']

for c in range(2):
    c_sil = np.sort(sil_vals[labels == c])
    size = c_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     facecolor=colors_sil[c], alpha=0.7, label=f'Cluster {c}')
    ax.text(-0.05, y_lower + 0.5 * size, str(c))
    y_lower = y_upper + 10

ax.axvline(silhouette_score(X_scaled, labels), color='red', linestyle='--',
           label=f'Score médio = {silhouette_score(X_scaled, labels):.3f}')
ax.set_xlabel('Silhouette Coefficient')
ax.set_title('Silhouette Plot por Amostra (K=2)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('\n💡 Barras longas e positivas = ponto bem alocado no cluster')
print('   Barras negativas = ponto provavelmente no cluster errado')

---
## 7. PCA como Ferramenta de Pré-processamento
Além de visualização, PCA pode ser usado para **reduzir dimensionalidade antes de treinar modelos**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

resultados_pca = {}
print('Comparando impacto do PCA na classificação:\n')

for n_comp in [2, 5, 10, 15, 20, 30]:
    if n_comp == 30:
        pipe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
        label = 'Sem PCA (30 features)'
    else:
        pipe = Pipeline([
            ('sc', StandardScaler()),
            ('pca', PCA(n_components=n_comp, random_state=42)),
            ('clf', LogisticRegression(max_iter=1000))
        ])
        label = f'PCA {n_comp} componentes'

    scores = cross_val_score(pipe, X, y_true, cv=5, scoring='f1')
    resultados_pca[label] = scores.mean()
    print(f'  {label:<30} F1 = {scores.mean():.4f} ± {scores.std():.4f}')

print('\n💡 Às vezes menos features (PCA) → modelo tão bom quanto com todas!')

---
## 8. Resumo

| Conceito | O que é | Quando usar |
|---|---|---|
| **K-Means** | Agrupa pontos por proximidade aos centroides | Segmentação sem labels |
| **Elbow Method** | Plota inércia vs K, busca o 'cotovelo' | Escolher K |
| **Silhouette Score** | Mede coesão e separação dos clusters | Validar qualidade do clustering |
| **PCA** | Comprime N features em K componentes (K < N) | Visualização, redução de ruído, velocidade |
| **ARI** | Compara clustering com ground truth | Quando você tem os labels para validar |